# Communication-Aware V2V Perception (Inference-First Kaggle Notebook)

- Repo: https://github.com/mohsenshahverdy/comm-aware-v2v-perception
- Kaggle: https://www.kaggle.com/code/mohsenshahverdi/communication-aware-v2v-perception/edit

This notebook is organized for **inference-first communication phase experiments** using existing checkpoint weights.


## 1. Environment setup


In [ ]:
print("Setting up virtual environment...")
!python -m pip install -U pip virtualenv
!virtualenv /kaggle/working/v2v_env
print("v2v_env created at /kaggle/working/v2v_env")


## 2. Install packages


In [ ]:
!/kaggle/working/v2v_env/bin/python -m pip install -U wrapt
!/kaggle/working/v2v_env/bin/python -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!/kaggle/working/v2v_env/bin/python -m pip install "cmake>=3.22" "easydict>=1.9" "tqdm>=4.64" "PyYAML>=6.0" "cython>=0.29.36"
!/kaggle/working/v2v_env/bin/python -m pip install "numpy>=1.24" "scipy>=1.11" "matplotlib>=3.7" "scikit-image>=0.22" "opencv-python>=4.8" "open3d>=0.18" "shapely>=2.0"
!/kaggle/working/v2v_env/bin/python -m pip install "torchviz>=0.0.2" "tensorboardX>=2.6" "einops>=0.7" "timm>=0.9"
!/kaggle/working/v2v_env/bin/python -m pip install --only-binary=:all: cumm-cu118 spconv-cu118


## 2.1 Verify runtime


In [ ]:
!/kaggle/working/v2v_env/bin/python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"
!/kaggle/working/v2v_env/bin/python -c "import spconv; print('spconv ok')"
!/kaggle/working/v2v_env/bin/python -c "import open3d; print('open3d ok')"
!/kaggle/working/v2v_env/bin/python -c "import yaml; print('yaml ok')"


## 3. Clone repo


In [ ]:
!cd /kaggle/working && git clone https://github.com/mohsenshahverdy/comm-aware-v2v-perception.git


## 4. Build extensions


In [ ]:
%cd /kaggle/working/comm-aware-v2v-perception
!/kaggle/working/v2v_env/bin/python src/utils/setup.py build_ext --inplace


## 5. Check datasets/checkpoint inputs


In [ ]:
!ls -lah /kaggle/input
!find /kaggle/input -maxdepth 2 -type d | head -100


## 5.1 Experiment variables


In [ ]:
from pathlib import Path

REPO_DIR = Path("/kaggle/working/comm-aware-v2v-perception")
ENV_PY = Path("/kaggle/working/v2v_env/bin/python")

DATA_ROOT = Path("/kaggle/input/data-all")
CHECKPOINT_INPUT = Path("/kaggle/input/best-epoch")

CARLA_TEST_DIR = DATA_ROOT / "test/test"
CULVER_TEST_DIR = DATA_ROOT / "test/test_culver_city/test_culver_city"

RUN_ROOT = Path("/kaggle/working/phase_runs")
RUN_ROOT.mkdir(exist_ok=True)

print("REPO_DIR:", REPO_DIR)
print("DATA_ROOT exists:", DATA_ROOT.exists(), DATA_ROOT)
print("CHECKPOINT_INPUT exists:", CHECKPOINT_INPUT.exists(), CHECKPOINT_INPUT)
print("CARLA_TEST_DIR exists:", CARLA_TEST_DIR.exists(), CARLA_TEST_DIR)
print("CULVER_TEST_DIR exists:", CULVER_TEST_DIR.exists(), CULVER_TEST_DIR)


## 6. Copy checkpoint + patch config helper


## 6.1 Check checkpoint contents


In [ ]:
!find /kaggle/input/best-epoch -maxdepth 3 -type f | head -50
!find /kaggle/input/best-epoch -name "*.pth" -o -name "config.yaml"


In [ ]:
import shutil
import re
from pathlib import Path

def prepare_phase_run(preset, split_name="carla"):
    run_dir = RUN_ROOT / f"{split_name}_{preset}"
    
    if run_dir.exists():
        shutil.rmtree(run_dir)
    shutil.copytree(CHECKPOINT_INPUT, run_dir)
    
    src_cfg = REPO_DIR / "src/hypes_yaml/point_pillar_intermediate_V2VAM.yaml"
    dst_cfg = run_dir / "config.yaml"
    shutil.copy(src_cfg, dst_cfg)

    # Also copy communication presets used by yaml_utils preset merge
    src_presets = REPO_DIR / "src/hypes_yaml/communication_phase_presets.yaml"
    dst_presets = run_dir / "communication_phase_presets.yaml"
    shutil.copy(src_presets, dst_presets)

    if split_name == "carla":
        validate_dir = CARLA_TEST_DIR
    elif split_name == "culver":
        validate_dir = CULVER_TEST_DIR
    else:
        raise ValueError(split_name)

    text = dst_cfg.read_text()

    # Set preset
    text = re.sub(
        r"communication_preset:\s*\S+",
        f"communication_preset: {preset}",
        text
    )

    # Robustly replace root/validate dirs
    text = re.sub(
        r'root_dir:\s*["\'].*?["\']',
        f'root_dir: "{DATA_ROOT / "train"}"',
        text
    )
    text = re.sub(
        r'validate_dir:\s*["\'].*?["\']',
        f'validate_dir: "{validate_dir}"',
        text
    )

    dst_cfg.write_text(text)

    print("Prepared:", run_dir)
    print("Preset:", preset)
    print("Split:", split_name)
    print("Config check:")
    !grep -n "communication_preset\|root_dir:\|validate_dir:" {dst_cfg}
    !ls -lah {run_dir} | head

    return run_dir


## 7. Inference runner helper


In [ ]:
def run_inference(run_dir, log_name="inference.log"):
    import subprocess, os

    help_cmd = f"cd {REPO_DIR} && PYTHONPATH={REPO_DIR} {ENV_PY} -m src.tools.inference --help"
    help_out = subprocess.getoutput(help_cmd)
    has_global_sort = "--global_sort_detections" in help_out

    extra = "--global_sort_detections" if has_global_sort else ""

    cmd = f"""
    cd {REPO_DIR} && \
    PYTHONPATH={REPO_DIR} \
    {ENV_PY} -u -m src.tools.inference \
      --model_dir {run_dir} \
      --fusion_method intermediate \
      {extra} \
      2>&1 | tee {run_dir / log_name}
    """
    print("global_sort_supported:", has_global_sort)
    print(cmd)
    return subprocess.call(cmd, shell=True)


In [ ]:
# Run only if inference fails with pcdet_utils/CUDA extension import error
#!/kaggle/working/v2v_env/bin/python src/pcdet_utils/setup.py build_ext --inplace


In [ ]:
!/kaggle/working/v2v_env/bin/python -m src.tools.inference --help


## 8. Run first CARLA checks (phase0/phase1 only)


In [ ]:
phase_order = [
    "phase0_baseline",
    "phase1_measurement",
]

for preset in phase_order:
    run_dir = prepare_phase_run(preset, split_name="carla")
    code = run_inference(run_dir, log_name=f"{preset}.log")
    print("Exit code:", code)
    if code != 0:
        raise RuntimeError(f"Failed at {preset}")


## 9. Run CARLA phase2 baselines (after phase0/phase1 pass)


In [ ]:
for preset in ["phase2_random_drop", "phase2_topk_energy", "phase2_neighbor_packetloss"]:
    run_dir = prepare_phase_run(preset, split_name="carla")
    code = run_inference(run_dir, log_name=f"{preset}.log")
    print("Exit code:", code)
    if code != 0:
        raise RuntimeError(f"Failed at {preset}")


## 10. Optional Culver runs (after CARLA phase2 works)


In [ ]:
for preset in ["phase0_baseline", "phase1_measurement", "phase2_topk_energy"]:
    run_dir = prepare_phase_run(preset, split_name="culver")
    code = run_inference(run_dir, log_name=f"{preset}.log")
    print("Exit code:", code)
    if code != 0:
        raise RuntimeError(f"Failed at {preset} on Culver")


## 11. Result inspection


In [ ]:
!find /kaggle/working/phase_runs -name "summary_eval.yaml" -o -name "comm_metrics_epoch.csv" -o -name "comm_metrics_frame.jsonl"


## 12. Plot metrics


In [ ]:
import subprocess
for csv in RUN_ROOT.glob("*/comm_metrics_epoch.csv"):
    print("Plotting:", csv)
    cmd = f"cd {REPO_DIR} && PYTHONPATH={REPO_DIR} {ENV_PY} -m src.tools.plot_comm_metrics --csv {csv}"
    subprocess.call(cmd, shell=True)


## Warning

- `phase0`, `phase1`, `phase2` use old checkpoint weights.
- `phase3_learnable_mask` and `phase4_repair` need training/fine-tuning before results are meaningful.


## Optional: Training / Fine-tuning

Do not run this section before finishing phase0/phase1/phase2 inference experiments.


In [ ]:
%cd /kaggle/working/comm-aware-v2v-perception

# Example single-GPU training/fine-tuning
!/kaggle/working/v2v_env/bin/python -u -m src.tools.train \
  --hypes_yaml src/hypes_yaml/point_pillar_intermediate_V2VAM.yaml \
  2>&1 | tee /kaggle/working/training.log
